# transistorの回転拡張：補完・切り出し・90度回転を比較する
2026-11-05改訂。**この改訂版はコード作成のみで、未実行です。旧版の出力は削除しました。** 上から順に実行してください。

| 比較の枠 | 方法 | 回転角度 | 回転後に残す範囲 |
| --- | --- | --- | --- |
| 同一条件で比較 | ①白埋め／②外周RGB中央値／③edge／④reflect | 36度刻み・0度を含む10方向 | 1024×1024 |
| 別条件の補足 | ⑤補完領域を除く中央切り出し | 同じ10方向 | 共通の中央720×720（初期設定） |
| 別条件の補足 | ⑥90度刻みの回転 | 0・90・180・270度の4方向 | 1024×1024 |

①〜④は元画像全体1024×1024、回転の最近傍補間、最終128×128へのLANCZOS縮小・白黒・0〜1を共通とし、枠外の埋め方だけを変えます。0度は元画像をそのまま使います。

⑤は見える範囲と縮小率が変わり、⑥は方向数が異なります。①〜④と同じ条件の比較として扱いません。

**画像加工の比較用です。学習・モデル推論・AUROC評価は含めません。学習用ノートブックは2026-11-06にreflect補完の方針へ変更しました。この比較用ノートブックは学習を行いません。見た目と異常検知性能は区別して判断します。**

In [ ]:
from pathlib import Path
from functools import lru_cache
import hashlib
import math
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import ListedColormap

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'memory/memory.md').is_file() and (p / 'data/transistor').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリ内を作業ディレクトリにしてください。')
TRAIN_DIR = ROOT / 'data/transistor/train/good'
PRIMARY_IMAGE = '000.png'
SAMPLE_IMAGES = ['000.png', '001.png', '002.png']  # 他の正常画像に変更できます。
ANGLES = tuple(range(0, 360, 36))
COMPARE_ANGLES = (36, 72, 144)
MULTI_ANGLES = (36, 72)
RIGHT_ANGLES = (0, 90, 180, 270)
MODES = ('white', 'background', 'edge', 'reflect')
LABELS = {'white': '① 白埋め', 'background': '② 外周RGB中央値',
          'edge': '③ edge：端を延長', 'reflect': '④ reflect：鏡写し'}
BACKGROUND_BORDER = 16  # 中央値を求める外周の幅。被写体の混入に注意。
ZOOM_BOX = (0, 0, 360, 360)  # 1024画像上の(x0, y0, x1, y1)。①〜④の同じ領域を拡大。
SAFE_CROP_SIDE = 720  # 初期値はfloor(1024/sqrt(2))-4。⑤は0度もこの範囲へ切り出す。
np.random.seed(42)
fonts = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams.update({'font.family': next((f for f in ['Meiryo', 'Yu Gothic', 'Noto Sans CJK JP']
                                         if f in fonts), 'DejaVu Sans'),
                     'font.size': 11, 'axes.unicode_minus': False,
                     'figure.dpi': 100, 'figure.facecolor': 'white'})
if not SAMPLE_IMAGES or not COMPARE_ANGLES or not MULTI_ANGLES:
    raise ValueError('比較画像と角度を1つ以上指定してください。')
if len(set(MODES)) != len(MODES):
    raise ValueError('補完方法が重複しています。')
source_names = list(dict.fromkeys([PRIMARY_IMAGE, *SAMPLE_IMAGES]))
images, source_hashes, background_colors = {}, {}, {}
for name in source_names:
    if Path(name).name != name:
        raise ValueError('画像名はtrain/good内のファイル名のみ指定してください。')
    path = TRAIN_DIR / name
    source_hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()
    with Image.open(path) as im:
        images[name] = im.convert('RGB').copy()
    if images[name].size != (1024, 1024):
        raise ValueError(f'想定外の画像サイズ：{name} {images[name].size}')
    if not 1 <= BACKGROUND_BORDER < min(images[name].size) / 2:
        raise ValueError('外周幅が不適切です。')
    pixels = np.asarray(images[name])
    border_mask = np.zeros(pixels.shape[:2], dtype=bool)
    border_mask[:BACKGROUND_BORDER, :] = True
    border_mask[-BACKGROUND_BORDER:, :] = True
    border_mask[:, :BACKGROUND_BORDER] = True
    border_mask[:, -BACKGROUND_BORDER:] = True
    background_colors[name] = tuple(int(v) for v in np.rint(np.median(pixels[border_mask], axis=0)))
x0, y0, x1, y1 = ZOOM_BOX
if not (0 <= x0 < x1 <= 1024 and 0 <= y0 < y1 <= 1024):
    raise ValueError('拡大範囲が画像の外に出ています。')
if not isinstance(SAFE_CROP_SIDE, int) or not 1 <= SAFE_CROP_SIDE <= 1024:
    raise ValueError('SAFE_CROP_SIDEは1〜1024の整数にしてください。')


def edge_pad_width(size):
    return math.ceil((math.hypot(*size) - min(size)) / 2) + 4


@lru_cache(maxsize=128)
def original_region(name, angle):
    image = images[name]
    return np.asarray(Image.new('L', image.size, 255).rotate(
        angle, resample=Image.Resampling.NEAREST, expand=False, fillcolor=0)) > 0


@lru_cache(maxsize=64)
def rotated(name, angle, mode):
    if mode not in MODES:
        raise ValueError(mode)
    image = images[name]
    width, height = image.size
    mask = original_region(name, angle)
    if angle == 0:
        return image.copy(), mask
    # どの方法でも元画像由来の画素を同じ値に保つ。
    original = np.asarray(image.rotate(angle, resample=Image.Resampling.NEAREST,
                                      expand=False, fillcolor=(255, 255, 255)))
    if mode in ('white', 'background'):
        fill = (255, 255, 255) if mode == 'white' else background_colors[name]
        result = np.asarray(image.rotate(angle, resample=Image.Resampling.NEAREST,
                                         expand=False, fillcolor=fill)).copy()
    else:
        pad = edge_pad_width(image.size)
        extended = Image.fromarray(np.pad(np.asarray(image),
                                          ((pad, pad), (pad, pad), (0, 0)), mode=mode))
        box = (pad, pad, pad + width, pad + height)
        result = np.asarray(extended.rotate(angle, resample=Image.Resampling.NEAREST,
                                            expand=False, fillcolor=(255, 255, 255)).crop(box)).copy()
        support = Image.new('L', extended.size, 255).rotate(
            angle, resample=Image.Resampling.NEAREST, expand=False, fillcolor=0).crop(box)
        assert np.all(np.asarray(support) == 255), '補完幅が不足しています。'
    result[mask] = original[mask]
    return Image.fromarray(result), mask


def final_input(image):
    small = image.resize((128, 128), resample=Image.Resampling.LANCZOS).convert('L')
    return np.asarray(small, dtype=np.float32) / np.float32(255)


def show_input(ax, array):
    ax.imshow(array, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.axis('off')


def final_zoom_box():
    return math.floor(x0 / 8), math.floor(y0 / 8), math.ceil(x1 / 8), math.ceil(y1 / 8)


def draw_zoom_box(ax):
    a, b, c, d = final_zoom_box()
    ax.add_patch(Rectangle((a - .5, b - .5), c - a, d - b,
                           fill=False, edgecolor='#dc6b40', linewidth=2))


def safe_crop_box(name):
    width, height = images[name].size
    left, top = (width - SAFE_CROP_SIDE) // 2, (height - SAFE_CROP_SIDE) // 2
    return left, top, left + SAFE_CROP_SIDE, top + SAFE_CROP_SIDE


@lru_cache(maxsize=32)
def cropped_without_fill(name, angle):
    # 白埋めは一時的な回転枠のみに使い、補完画素を含まない範囲だけを残す。
    full, mask = rotated(name, angle, 'white')
    left, top, right, bottom = safe_crop_box(name)
    if not mask[top:bottom, left:right].all():
        raise ValueError(f'{angle}度の切り出しに補完画素が含まれます。SAFE_CROP_SIDEを小さくしてください。')
    return full.crop((left, top, right, bottom))


def quarter_rotated(name, angle):
    if angle not in RIGHT_ANGLES:
        raise ValueError('90度刻みの角度を指定してください。')
    if angle == 0:
        return images[name].copy()
    operation = {90: Image.Transpose.ROTATE_90,
                 180: Image.Transpose.ROTATE_180,
                 270: Image.Transpose.ROTATE_270}[angle]
    return images[name].transpose(operation)


print('対象画像：', source_names, '/ edge・reflectの補完幅：', edge_pad_width((1024, 1024)), '画素')
print('外周RGB中央値：', background_colors, '/ 拡大範囲：', ZOOM_BOX)
print('①〜④は同条件、⑤は切り出し範囲・縮小率、⑥は方向数が異なります。')

## 1. 元画像と、外周から求めた背景色
外周16画素のRGB中央値を画像ごとに求めます。黒い丸や線が外周にあるため、中央値は必ずしも純粋な背景色ではありません。色が近くても、丸・線の模様は単色では再現できません。

In [ ]:
fig, axes = plt.subplots(1, len(source_names), figsize=(5 * len(source_names), 5),
                         squeeze=False, constrained_layout=True)
for ax, name in zip(axes.flat, source_names):
    ax.imshow(images[name])
    ax.set_title(f'{name} / 元画像1024×1024\n外周RGB中央値：{background_colors[name]}')
    ax.axis('off')
plt.show()
plt.close(fig)

## 2. ①〜④を同じ角度で比較：最終128×128入力
上から白埋め、外周RGB中央値、edge、reflectです。橙色の枠は次の拡大図の領域です。回転角度・補間・画像サイズ・縮小・白黒化は共通です。

In [ ]:
fig, axes = plt.subplots(len(MODES), len(COMPARE_ANGLES),
                         figsize=(12, 3.5 * len(MODES)), squeeze=False, constrained_layout=True)
for row, mode in enumerate(MODES):
    for col, angle in enumerate(COMPARE_ANGLES):
        show_input(axes[row, col], final_input(rotated(PRIMARY_IMAGE, angle, mode)[0]))
        draw_zoom_box(axes[row, col])
        axes[row, col].set_title(f'{LABELS[mode]} / {angle}度')
fig.suptitle(f'{PRIMARY_IMAGE}：①〜④の最終入力を同条件で比較', fontsize=16)
plt.show()
plt.close(fig)

## 3. 画像端の拡大：縮小前の高解像度画像
各方法で同じ1024画像上の範囲を切り出します。単色部分と模様の境目、黒い丸の引き伸ばし、鏡写しによる折り返しを確認します。

In [ ]:
fig, axes = plt.subplots(len(MODES), len(COMPARE_ANGLES),
                         figsize=(12, 3.5 * len(MODES)), squeeze=False, constrained_layout=True)
for row, mode in enumerate(MODES):
    for col, angle in enumerate(COMPARE_ANGLES):
        image = rotated(PRIMARY_IMAGE, angle, mode)[0].crop(ZOOM_BOX).convert('L')
        axes[row, col].imshow(image, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
        axes[row, col].axis('off')
        axes[row, col].set_title(f'{LABELS[mode]} / {angle}度')
fig.suptitle(f'縮小前の画像端を拡大 / 範囲{ZOOM_BOX}', fontsize=16)
plt.show()
plt.close(fig)

## 4. 最終入力の画像端を拡大する
128×128へ縮小した後、対応する領域を拡大します。表示時の補間はnearestに固定し、追加のぼかしは加えません。

In [ ]:
a, b, c, d = final_zoom_box()
fig, axes = plt.subplots(len(MODES), len(COMPARE_ANGLES),
                         figsize=(12, 3.5 * len(MODES)), squeeze=False, constrained_layout=True)
for row, mode in enumerate(MODES):
    for col, angle in enumerate(COMPARE_ANGLES):
        array = final_input(rotated(PRIMARY_IMAGE, angle, mode)[0])[b:d, a:c]
        show_input(axes[row, col], array)
        axes[row, col].set_title(f'{LABELS[mode]} / {angle}度')
fig.suptitle(f'最終128×128入力の端 / 切り出し{c - a}×{d - b}画素', fontsize=16)
plt.show()
plt.close(fig)

## 5. 元画像の領域と補完領域
①〜④で補完する領域の位置は共通です。青緑は元画像由来、橙は枠外から補った領域です。これは画像枠のマスクであり、被写体や欠陥の正解マスクではありません。領域率も画質・検出性能の指標ではありません。

In [ ]:
fig, axes = plt.subplots(1, len(COMPARE_ANGLES), figsize=(12, 4.5),
                         squeeze=False, constrained_layout=True)
cmap = ListedColormap(['#e6ad72', '#418b8b'])
for ax, angle in zip(axes.flat, COMPARE_ANGLES):
    mask = original_region(PRIMARY_IMAGE, angle)
    ax.imshow(mask, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'{angle}度 / 補完領域 {100 * (~mask).mean():.1f}%')
    ax.axis('off')
fig.legend(handles=[Patch(color='#418b8b', label='元画像由来'),
                    Patch(color='#e6ad72', label='補完領域')],
           loc='lower center', ncol=2, bbox_to_anchor=(.5, .005))
fig.set_layout_engine('constrained', rect=(0, .12, 1, .88))
plt.show()
plt.close(fig)

## 6. 方法間の画素差：edgeを基準に比較する
白埋め・外周RGB中央値・reflectを、それぞれedgeと比較します。表示する値はRGB画素差の最大値（0〜255）です。**差が大きいほど良い・悪いという意味ではありません。** 高解像度の元画像領域では、どの方法も同じ画素を使うことを確認するための図です。

LANCZOS縮小後はフィルタが境界をまたぐため、元画像領域側の境界近くにも補完方法の違いが広がることがあります。

In [ ]:
DIFFERENCE_MODES = ('white', 'background', 'reflect')
fig, axes = plt.subplots(len(DIFFERENCE_MODES), len(COMPARE_ANGLES),
                         figsize=(12, 3.5 * len(DIFFERENCE_MODES)), squeeze=False, constrained_layout=True)
for row, mode in enumerate(DIFFERENCE_MODES):
    for col, angle in enumerate(COMPARE_ANGLES):
        edge, mask = rotated(PRIMARY_IMAGE, angle, 'edge')
        candidate, mask2 = rotated(PRIMARY_IMAGE, angle, mode)
        assert np.array_equal(mask, mask2)
        difference = np.abs(np.asarray(candidate).astype(np.int16)
                            - np.asarray(edge).astype(np.int16)).max(axis=2)
        assert np.all(difference[mask] == 0)
        heat = axes[row, col].imshow(difference, cmap='magma', vmin=0, vmax=255,
                                      interpolation='nearest')
        axes[row, col].set_title(f'{mode}とedgeの差 / {angle}度')
        axes[row, col].axis('off')
fig.colorbar(heat, ax=axes.ravel().tolist(), shrink=.75, label='RGB画素差の最大値（0〜255）')
plt.show()
plt.close(fig)

## 7. ①〜④の最終10方向を一覧表示
方法ごとに独立した図を作ります。0度を含む10方向で、0度は共通の通常入力です。

In [ ]:
for mode in MODES:
    fig, axes = plt.subplots(2, 5, figsize=(13, 6), constrained_layout=True)
    for ax, angle in zip(axes.flat, ANGLES):
        show_input(ax, final_input(rotated(PRIMARY_IMAGE, angle, mode)[0]))
        ax.set_title(f'{angle}度')
    fig.suptitle(f'{PRIMARY_IMAGE} / {LABELS[mode]} / 最終128×128・白黒', fontsize=16)
    plt.show()
    plt.close(fig)

## 8. 複数の正常画像で①〜④を比較する
画像ごとに図を分け、行を角度、列を4つの方法にします。設定セルのSAMPLE_IMAGESとMULTI_ANGLESを変更できます。

In [ ]:
for name in SAMPLE_IMAGES:
    fig, axes = plt.subplots(len(MULTI_ANGLES), len(MODES),
                             figsize=(14, 3.5 * len(MULTI_ANGLES)), squeeze=False,
                             constrained_layout=True)
    for row, angle in enumerate(MULTI_ANGLES):
        for col, mode in enumerate(MODES):
            show_input(axes[row, col], final_input(rotated(name, angle, mode)[0]))
            axes[row, col].set_title(f'{LABELS[mode]} / {angle}度')
    fig.suptitle(f'{name}：方法ごとの背景・対象・境界を確認', fontsize=16)
    plt.show()
    plt.close(fig)

## 9. 補足⑤：補完領域を含まない中央を切り出す
正方形の元画像で全方向に共通の切り出し範囲を使うため、初期設定は一辺720画素です。1024/√2の切り下げ値724より、画素の境界に余裕を取って4画素小さくしています。各角度で、切り出し内のマスクがすべて元画像由来であることをコードで確認します。範囲を変更して補完画素が入る場合はエラーにします。

**0度も720×720に切り出します。①〜④と0度の入力は一致しません。** 本体や端子が欠ける可能性があり、128×128へ縮小したときに対象が相対的に大きくなります。これを後で学習に採用するなら、通常条件やテスト画像の前処理も別途整理する必要があります。

上段は取り出す範囲（緑枠）、下段は切り出した最終入力です。上段の枠外にある白埋め部分は、下段には残しません。

In [ ]:
fig, axes = plt.subplots(2, len(COMPARE_ANGLES), figsize=(12, 8),
                         squeeze=False, constrained_layout=True)
for col, angle in enumerate(COMPARE_ANGLES):
    full, _ = rotated(PRIMARY_IMAGE, angle, 'white')
    axes[0, col].imshow(full)
    left, top, right, bottom = safe_crop_box(PRIMARY_IMAGE)
    axes[0, col].add_patch(Rectangle((left - .5, top - .5), right - left, bottom - top,
                                    fill=False, edgecolor='#22aa66', linewidth=2))
    axes[0, col].set_title(f'{angle}度 / 緑枠を取り出す')
    axes[0, col].axis('off')
    show_input(axes[1, col], final_input(cropped_without_fill(PRIMARY_IMAGE, angle)))
    axes[1, col].set_title(f'⑤ 中央{SAFE_CROP_SIDE}×{SAFE_CROP_SIDE} → 128×128')
fig.suptitle('補完なしの中央切り出し：対象が欠けないか確認', fontsize=16)
plt.show()
plt.close(fig)

fig, axes = plt.subplots(2, 5, figsize=(13, 6), constrained_layout=True)
for ax, angle in zip(axes.flat, ANGLES):
    show_input(ax, final_input(cropped_without_fill(PRIMARY_IMAGE, angle)))
    ax.set_title(f'{angle}度')
fig.suptitle(f'⑤ 中央切り出しの最終10方向 / 切り出し範囲は全方向で共通', fontsize=16)
plt.show()
plt.close(fig)

## 10. 補足⑥：90度刻みなら枠外の補完を避けられる
正方形の画像を0・90・180・270度に回す場合、画素の並べ替えで元の画像全体を保持できます。回転段階の補間・枠外の補完は行いません。最終128×128への縮小には他の方法と同じLANCZOSを使います。

**4方向なので、36度刻みの10方向と同じ拡張枚数にはなりません。** 正常として妥当な向きかは、補完の有無とは別に確認が必要です。

In [ ]:
fig, axes = plt.subplots(1, len(RIGHT_ANGLES), figsize=(14, 4),
                         squeeze=False, constrained_layout=True)
for ax, angle in zip(axes.flat, RIGHT_ANGLES):
    show_input(ax, final_input(quarter_rotated(PRIMARY_IMAGE, angle)))
    ax.set_title(f'⑥ {angle}度 / 補完なし')
fig.suptitle(f'{PRIMARY_IMAGE}：90度刻みの最終4方向', fontsize=16)
plt.show()
plt.close(fig)

## 11. ユーザー実行時の確認セル
以下の確認処理も、アシスタント側では実行していません。①〜④の元画像領域一致・0度一致、⑤の補完画素除外、⑥の画素保持、最終サイズ・値域、元画像のSHA-256を確認するコードです。

In [ ]:
for name in source_names:
    original = final_input(images[name])
    for mode in MODES:
        assert np.array_equal(final_input(rotated(name, 0, mode)[0]), original)
    check_angles = sorted(set(ANGLES + COMPARE_ANGLES + MULTI_ANGLES))
    for angle in check_angles:
        reference, mask = rotated(name, angle, 'white')
        for mode in MODES:
            result, current_mask = rotated(name, angle, mode)
            assert result.size == (1024, 1024) and np.array_equal(mask, current_mask)
            assert np.array_equal(np.asarray(result)[mask], np.asarray(reference)[mask])
            if angle != 0 and mode in ('white', 'background'):
                expected_fill = (255, 255, 255) if mode == 'white' else background_colors[name]
                assert np.all(np.asarray(result)[~mask] == np.asarray(expected_fill))
            small = final_input(result)
            assert small.shape == (128, 128) and small.dtype == np.float32
            assert np.isfinite(small).all() and 0 <= small.min() <= small.max() <= 1
        cropped = cropped_without_fill(name, angle)
        assert cropped.size == (SAFE_CROP_SIDE, SAFE_CROP_SIDE)
    original_pixels = np.asarray(images[name])
    for angle in RIGHT_ANGLES:
        result = quarter_rotated(name, angle)
        assert result.size == (1024, 1024)
        expected = np.rot90(original_pixels, k=angle // 90)
        assert np.array_equal(np.asarray(result), expected)
        small = final_input(result)
        assert small.shape == (128, 128) and np.isfinite(small).all()
        assert 0 <= small.min() <= small.max() <= 1
    assert hashlib.sha256((TRAIN_DIR / name).read_bytes()).hexdigest() == source_hashes[name]
print('ユーザー実行時の確認完了：領域・0度・サイズ・値域・90度画素保持・原画像SHA-256')
print(json.dumps({'primary_image': PRIMARY_IMAGE, 'sample_images': SAMPLE_IMAGES,
                  'same_condition_methods': MODES, 'angles': ANGLES,
                  'background_border': BACKGROUND_BORDER, 'background_colors': background_colors,
                  'edge_reflect_pad': edge_pad_width((1024, 1024)),
                  'rotation_interpolation': 'NEAREST', 'resize': 'LANCZOS',
                  'safe_crop_side': SAFE_CROP_SIDE, 'right_angles': RIGHT_ANGLES,
                  'network_input': [128, 128], 'source_sha256': source_hashes},
                 ensure_ascii=False, indent=2))

## 比較して決めること
- ①白埋め：白い三角形と元の背景の境目が目立たないか。
- ②外周RGB中央値：色は近づいても、背景の丸や線が途切れていないか。
- ③edge：黒い丸や線が帯状に伸びていないか。
- ④reflect：引き伸ばしの代わりに、模様の折り返しや対象の複製が起きていないか。
- ⑤中央切り出し：本体・端子の欠け、見える範囲・対象の大きさの変化を許容できるか。
- ⑥90度刻み：4方向で目的を満たすか。補完がなくても、正常画像として妥当な向きか。

この比較用ノートブック自体は学習処理を変更しません。2026-11-06にユーザー方針でreflectを採用し、別の学習用ノートブックをreflect補完へ変更しています。見た目の比較だけではAUROC改善は判断できません。

参照：[現在のtransistor学習ノートブック](261105transistorの通常学習とreflect回転拡張の比較学習とAUROC評価.ipynb)、[NumPy公式numpy.pad](https://numpy.org/doc/stable/reference/generated/numpy.pad.html)。